# 12_跨部門_請購驗收核對

跨工作表核對與明細報表

任務：以請購單號連結「資料」與「驗收」工作表，不能依列位置拼接。；先加總同一請購單的分批驗收數量，保留沒有驗收的請購單。；計算未到量與未到金額，輸出核對報表。

規則：每張請購單只有一種品項；同單可分批驗收。沒有驗收紀錄才視為收到 0 件；未到金額＝未到量×單價，不代表應付款。

先人工核對再執行。資料皆為教學模擬值。

In [ ]:
%pip -q install openpyxl numpy matplotlib

In [ ]:
from pathlib import Path
import csv
from openpyxl import load_workbook

# 本機：把 Notebook 與 Excel 放在同一資料夾。Colab：先上傳本例 Excel。
SOURCE = Path('12_跨部門_請購驗收核對.xlsx')
if not SOURCE.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f'請將 {SOURCE.name} 放到目前工作目錄：{Path.cwd()}')
    files.upload()
if not SOURCE.exists():
    raise FileNotFoundError(f'尚未找到 {SOURCE.name}，請確認上傳檔名')
OUT = Path('output') / '12_跨部門_請購驗收核對'
OUT.mkdir(parents=True, exist_ok=True)

def read_rows(sheet):
    data = sheet.iter_rows(values_only=True)
    headers = next(data)
    return [dict(zip(headers, row)) for row in data if any(v is not None for v in row)]

def save_csv(name, records):
    if not records:
        print('沒有符合條件的資料，未輸出 CSV')
        return
    with (OUT / name).open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)

# 來源檔只有原始值，沒有需要重算的 Excel 公式。
wb = load_workbook(SOURCE, read_only=True, data_only=True)
print('工作表：', wb.sheetnames)
rows = read_rows(wb['資料'])
print('第一筆：', rows[0])

In [ ]:
from collections import defaultdict
receipts = read_rows(wb['驗收'])
quantity = defaultdict(int)
request_ids = {r['請購單號'] for r in rows}
assert len(request_ids) == len(rows), '請購主檔單號重複'
for r in receipts:
    assert r['請購單號'] in request_ids, '驗收有未知請購單號'
    quantity[r['請購單號']] += r['驗收數量']
result = []
for r in rows:
    received = quantity[r['請購單號']]
    assert received <= r['請購數量'], '有超收，需另外查核'
    missing = r['請購數量'] - received
    result.append(dict(r, 已到量=received, 未到量=missing, 未到金額_元=missing*r['單價_元']))
print(result)
save_csv('請購驗收核對.csv', result)
assert [r['未到量'] for r in result] == [0,10,10,0]
assert sum(r['未到金額_元'] for r in result) == 700
wb.close()

## 驗收答案

PR001 分批 12＋8＝20 件；PR002 未到 10 件、200 元；PR003 未到 10 件、500 元；合計 700 元。